# Lab 3 — Exception Handling In FastAPI

Difficulty: Beginner | ~35-40 min 

### Step 1: Install Dependencies

We install the exact pinned versions of every library this lab needs. Run this cell first so everything is available for the rest of the notebook.

In [1]:
!pip install fastapi==0.112.2 pydantic==2.8.2 httpx==0.28.1 google-genai==1.29.0 python-dotenv==1.2.3


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 2: Imports and API key setup.

We import the modules we need and load the Google API key from a `.env` file. The key is stored as a string — the actual `genai.Client` instances are created inside the endpoint so each request gets a fresh client (this avoids event-loop binding issues with `TestClient`).

In [2]:
from fastapi import FastAPI, status
from fastapi.responses import JSONResponse
from fastapi.requests import Request
from fastapi.testclient import TestClient
from dotenv import load_dotenv
from google import genai
from google.genai import errors
import asyncio
import os

# Load GOOGLE_API_KEY from .env file in the current directory
load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    api_key = input("Enter your Google API key: ")

app = FastAPI()

### Step 3: Custom exception classes.

Each class represents one distinct upstream failure that can happen when calling an LLM provider. We define them so that FastAPI can catch each one by type and return a structured error response with the right HTTP status code.

In [3]:
class LLMAuthError(Exception):
    """Raised when the Gemini API rejects the API key (invalid credentials)."""
    pass

class LLMSafetyError(Exception):
    """Raised when the response is technically successful but contains no usable content (safety block)."""
    pass

class LLMTimeoutError(Exception):
    """Raised when the Gemini API does not respond within the allowed time."""
    pass

class BadModelError(Exception):
    """Raised when the requested model name is invalid or has been deprecated."""
    pass


### Step 4: Exception handlers registered on the app.

`@app.exception_handler(ExceptionType)` tells FastAPI: "whenever this exception type is raised anywhere in the request lifecycle, call this function and return its response instead of crashing." Each handler below maps one custom exception to a specific HTTP status code and error message.

In [4]:
# 401 Unauthorized — the API key was rejected by Gemini
@app.exception_handler(LLMAuthError)
async def auth_error_handler(request: Request, exc: LLMAuthError):
    return JSONResponse(
        status_code = status.HTTP_401_UNAUTHORIZED,
        content = {
            "error": "authorization error",
            "message": "Unable to authenticate with Gemini API"
        }
    )

In [5]:
# 504 Gateway Timeout — the upstream LLM took too long to respond
@app.exception_handler(LLMTimeoutError)
async def timeout_handler(request: Request, exc: LLMTimeoutError):
    return JSONResponse(
        status_code = status.HTTP_504_GATEWAY_TIMEOUT,
        content = {
            "error": "Request Timed Out",
            "message": "Gemini API timed out"
        }
    )

In [6]:
# 502 Bad Gateway — the API call succeeded but the content is unusable (safety block)
@app.exception_handler(LLMSafetyError)
async def safety_error(request: Request, exc: LLMSafetyError):
    return JSONResponse(
        status_code= status.HTTP_502_BAD_GATEWAY,
        content = {
            "error": "unsafe content",
            "message": "unusable content returned"
        }
    )

In [7]:
# 404 Not Found — the model name is invalid or deprecated
@app.exception_handler(BadModelError)
async def bad_model(request: Request, exc: BadModelError):
    return JSONResponse(
        status_code = status.HTTP_404_NOT_FOUND,
        content = {
            "error": "Model Not found",
            "message": "Model name is not valid"
        }
    )

### Step 5: The `/chat` endpoint — one try/except, four failure paths.

This single endpoint handles everything: a normal call, plus four ways it can fail upstream. Each boolean query parameter simulates a different failure:
- `deprecated_model=True` → invalid model name → `ClientError` code 404 → `BadModelError`
- `timeout_error=True` → artificially short timeout → `asyncio.TimeoutError` → `LLMTimeoutError`
- `bad_key=True` → invalid API key → `ClientError` code 400 → `LLMAuthError`
- `unsafe_content=True` → directly raises `LLMSafetyError` (no SDK exception — simulates a safety block where the API succeeds but returns nothing usable)


In [8]:
@app.post("/chat")
async def chat(
    message: str, 
    deprecated_model: bool = False, 
    timeout_error: bool = False,
    bad_key: bool = False,
    unsafe_content: bool= False):
    try:

        # bad_key=True uses a deliberately invalid key to simulate an auth failure
        active_client = genai.Client(api_key="Invalid_API_Key") if bad_key else genai.Client(api_key=api_key)

        # Pick model name and timeout based on failure flags
        model_name = "invalid_model_name" if deprecated_model else "gemini-2.5-flash" 
        timeout = 0.1 if timeout_error else 20

        # Call Gemini with a hard timeout
        response = await asyncio.wait_for(
            active_client.aio.models.generate_content(
                model= model_name,
                contents= message,
            ),
        timeout= timeout
        )


    except errors.ClientError as e:
        if e.code == 404:
            raise BadModelError()
        if e.code == 400:
            raise LLMAuthError()
        
        return JSONResponse(
            status_code=e.code,
            content={
                "details": e.details
        }
    ) #catches unhandled errors

    except asyncio.TimeoutError as e:
        raise LLMTimeoutError()
    
    # Simulate a safety block: raise directly (no SDK exception to catch)
    if unsafe_content:
        raise LLMSafetyError()

    return {"answer": response.text}

### Step 6: Demonstrations.

We first confirm that a normal call works and returns a real answer. Then we trigger each of the four failure modes one by one, showing that each returns the expected status code and structured error message.

Note: If 'Event Loop Closed' errors occurs, try rerunning the cell again. It will be fixed.

In [9]:
test_client = TestClient(app)

#### Case 1: Success case — no failure flags.

A completely normal call. This confirms that the exception handlers do not interfere with the happy path — they only activate when something actually goes wrong upstream.

In [10]:
# Success case: normal call, no failure flags
question = "What is the capital of France?"
res = test_client.post(
    "/chat",
    params={"message": question}
)
print(res.status_code)
print(res.json())

200
{'answer': 'The capital of France is **Paris**.'}


#### Case 2: Bad model — `deprecated_model=True`

Passes an invalid model name to Gemini, which returns a `ClientError` with code 404. The endpoint catches it and raises `BadModelError`, which the app-level handler turns into a `404 Not Found` response.

In [11]:
# Deprecated/bad model name -> 404
question = "hello"
res = test_client.post(
    "/chat",
    params={
        "message": question,
        "deprecated_model": True
    }
)

print(res.status_code)
print(res.json())

404
{'error': 'Model Not found', 'message': 'Model name is not valid'}


#### Case 3: Timeout — `timeout_error=True`

Sets a 0.1-second timeout on a call that would normally take much longer. The `asyncio.TimeoutError` is caught and re-raised as `LLMTimeoutError`, producing a `504 Gateway Timeout`.

In [12]:
# Timeout -> 504
question = "hello"
res = test_client.post(
    "/chat",
    params={
        "message": question,
        "timeout_error": True
    }
)

print(res.status_code)
print(res.json())

504
{'error': 'Request Timed Out', 'message': 'Gemini API timed out'}


#### Case 4: Unsafe content — `unsafe_content=True`

The call itself succeeds (the Gemini API responds), but we directly raise `LLMSafetyError()` to simulate what happens when a real safety filter silently blocks the content. No SDK exception is raised here — the handler is triggered purely by the application-level raise.

In [13]:
# Unsafe content -> 502
question = "hello"
res = test_client.post(
    "/chat",
    params={
        "message": question,
        "unsafe_content": True
    }
)

print(res.status_code)
print(res.json())

502
{'error': 'unsafe content', 'message': 'unusable content returned'}


#### Case 5: Bad key — `bad_key=True`

Creates a client with a deliberately invalid API key. Gemini returns a `ClientError` with code 400, which the endpoint catches and re-raises as `LLMAuthError`, producing a `401 Unauthorized`.

In [14]:
# Bad API key -> 401
question = "hello"
res = test_client.post(
    "/chat",
    params={
        "message": question,
        "bad_key": True
    }
)

print(res.status_code)
print(res.json())

401
{'error': 'authorization error', 'message': 'Unable to authenticate with Gemini API'}
